# Dự báo Doanh thu Sản phẩm bằng Hồi quy Gradient Boosting
## 1. Giới thiệu tổng quan

### Tổng quan Dự án
Dự án này xây dựng một quy trình hồi quy hoàn chỉnh từ đầu đến cuối để dự báo **Doanh thu Sản phẩm** (`Sales_Revenue`) dựa trên dữ liệu giao dịch của khách hàng, sử dụng các thuật toán **Gradient Boosting** phổ biến bao gồm HistGradientBoosting (của Scikit-learn), XGBoost, LightGBM và CatBoost.

| Bước | Notebook | Mô tả |
|------|----------|-------|
| 1 | `01_introduction.ipynb` | Tổng quan dự án và phát biểu bài toán |
| 2 | `02_data_checks.ipynb` | Kiểm tra chất lượng và tính toàn vẹn của dữ liệu |
| 3 | `03_data_cleaning.ipynb` | Làm sạch dữ liệu và chuẩn bị biến mục tiêu |
| 4 | `04_eda.ipynb` | Phân tích và khám phá dữ liệu chi tiết |
| 5 | `05_feature_engineering.ipynb` | Trích xuất đặc trưng, mã hóa, chuẩn hóa và phân tách dữ liệu |
| 6 | `06_model_training.ipynb` | Huấn luyện các mô hình hồi quy Gradient Boosting |
| 7 | `07_evaluation.ipynb` | Xây dựng các hàm đánh giá từ đầu và so sánh các mô hình |
| 8 | `08_conclusion.ipynb` | Tổng kết kết quả và định hướng tối ưu hóa dữ liệu |

### Mục tiêu Học tập
- Thực hiện kiểm tra dữ liệu giao dịch thực tế để phát hiện các giá trị bất hợp lý và vấn đề về thời gian thu thập dở dang.
- Phân tích khám phá dữ liệu trực quan sâu sắc về xu hướng phân phối doanh thu, các nhóm khách hàng và hành vi tiêu dùng thời gian qua.
- Xây dựng quy trình tiền xử lý đặc trưng chuẩn xác bao gồm trích xuất ngày tháng, chuẩn hóa phân phối và mã hóa biến phân loại.
- Tự cài đặt các chỉ số đánh giá hồi quy (MAE, MSE, RMSE, R-squared) từ đầu bằng Python/NumPy.
- Huấn luyện, đánh giá và so sánh nhiều thuật toán Gradient Boosting hiện đại, bao gồm cả việc tự xây dựng thuật toán LightGBM thu gọn từ đầu.
- Định hướng tối ưu hóa hệ thống dựa trên đặc tính của mô hình chiến thắng.

### Thông tin Bộ dữ liệu
- **Đường dẫn:** `../../data/raw-data/customer_shopping_data.csv`
- **Tổng số dòng:** 99.457
- **Các cột dữ liệu:** `invoice_no` (mã hóa đơn), `customer_id` (mã khách hàng), `gender` (giới tính), `age` (tuổi), `category` (danh mục sản phẩm), `quantity` (số lượng), `price` (đơn giá), `payment_method` (phương thức thanh toán), `invoice_date` (ngày xuất hóa đơn), `shopping_mall` (trung tâm thương mại).

### Cơ sở lựa chọn mô hình
Chúng ta sẽ so sánh và đánh giá 5 mô hình hồi quy:
- **HistGradientBoostingRegressor (Sklearn):** Hỗ trợ tốt dữ liệu lớn, phân tách nhánh dựa trên các khoảng histogram.
- **XGBRegressor (XGBoost):** Tối ưu hóa cao độ, tích hợp các kỹ thuật điều chuẩn (regularization) tránh quá khớp.
- **LGBMRegressor (LightGBM):** Phát triển cây theo chiều sâu (leaf-wise), tốc độ nhanh, tiết kiệm bộ nhớ.
- **CatBoostRegressor (CatBoost):** Xử lý cực tốt các biến chữ phân loại nguyên bản mà không cần mã hóa trước.
- **LightGBM tự build (Scratch):** Phiên bản tự cài đặt tích hợp phân nhóm Histogram và phát triển cây theo chiều sâu (leaf-wise) để hiểu rõ cơ chế hoạt động.

In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

RAW_DATA_PATH = r'../../data/raw-data/customer_shopping_data.csv'
df = pd.read_csv(RAW_DATA_PATH)
print('Kích thước tập dữ liệu:', df.shape)
print('Các cột dữ liệu        :', list(df.columns))
df.head()

Kích thước tập dữ liệu: (99457, 10)
Các cột dữ liệu        : ['invoice_no', 'customer_id', 'gender', 'age', 'category', 'quantity', 'price', 'payment_method', 'invoice_date', 'shopping_mall']


,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,5/8/2022,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,12/12/2021,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,9/11/2021,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,16/05/2021,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,24/10/2021,Kanyon
